# Notebook 02: Huấn luyện Phân tán Dual-GPU DDP trên Kaggle T4 ×2
**Dự án:** Vietnamese ASR → English Translation trên Kaggle T4 ×2

### Mục tiêu:
1. Khởi tạo môi trường DDP với `torchrun --nproc_per_node=2`.
2. Chạy kiểm tra Smoke test và Pilot test (100 updates) xác nhận không OOM trên 2× T4 (Gate G4, G6).
3. Huấn luyện 3 biến thể đối chứng công bằng (cùng initial weights, cùng effective batch 32, cùng schedule):
   - **B1**: Baseline ViP-VL + Pure Linear CTC fine-tuning.
   - **P1**: B1 + Token-Frame Attention Pooling (\(\lambda=0.1\)).
   - **P2**: P1 + Soft Gaussian Temporal Prior (\(\beta=0.1, \sigma=0.2\)).
4. Lưu checkpoint nguyên tử (Atomic Checkpoint) và lưu vết huấn luyện.

In [ ]:
# 1. Chẩn đoán 2x GPU T4
!nvidia-smi
import torch
assert torch.cuda.device_count() >= 2, f"Cần ít nhất 2 GPU T4, hiện có: {torch.cuda.device_count()}"
print(f"Đã phát hiện {torch.cuda.device_count()} GPU: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

In [ ]:
# 2. Chuyển vào thư mục dự án và kiểm tra CLI
%cd /kaggle/working/vi_asr_project
!python -m vi_asr --help

In [ ]:
# 3. Chạy Smoke Test (Gate G4: Kiểm chứng hội tụ học trên toy dataset)
!python -m vi_asr smoke \
    --config configs/base.yaml \
    --paths configs/paths/kaggle.yaml

In [ ]:
# 4. Chạy Pilot Run (Gate G6: 100 updates trên Dual GPU T4x2 đo VRAM & throughput)
!torchrun --nproc_per_node=2 -m vi_asr pilot \
    --config configs/base.yaml \
    --variant configs/variants/p2.yaml \
    --profile configs/profiles/t4x2.yaml \
    --paths configs/paths/kaggle.yaml \
    --optimizer-updates 100 \
    --run-dir runs/pilot_p2

In [ ]:
# 5. Huấn luyện Biến thể B1 (Baseline ViP-VL + Pure Linear CTC)
!torchrun --nproc_per_node=2 -m vi_asr train \
    --config configs/base.yaml \
    --variant configs/variants/b1.yaml \
    --profile configs/profiles/t4x2.yaml \
    --paths configs/paths/kaggle.yaml \
    --run-dir runs/b1_ctc \
    --seed 42

In [ ]:
# 6. Huấn luyện Biến thể P1 (ViP-VL + CTC + Token-Frame Pooling)
!torchrun --nproc_per_node=2 -m vi_asr train \
    --config configs/base.yaml \
    --variant configs/variants/p1.yaml \
    --profile configs/profiles/t4x2.yaml \
    --paths configs/paths/kaggle.yaml \
    --run-dir runs/p1_token_pooling \
    --seed 42

In [ ]:
# 7. Huấn luyện Biến thể P2 (ViP-VL + CTC + Token-Frame Pooling + Temporal Prior)
!torchrun --nproc_per_node=2 -m vi_asr train \
    --config configs/base.yaml \
    --variant configs/variants/p2.yaml \
    --profile configs/profiles/t4x2.yaml \
    --paths configs/paths/kaggle.yaml \
    --run-dir runs/p2_token_temporal \
    --seed 42